In [1]:
import yfinance as yf

data = yf.download("AAPL", start="2020-01-01", end="2024-01-01")
data = data[['Close']]
print(data.head())

/tmp/ipykernel_17303/2031979924.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download("AAPL", start="2020-01-01", end="2024-01-01")
[*********************100%***********************]  1 of 1 completed

Price           Close
Ticker           AAPL
Date                 
2020-01-02  72.400505
2020-01-03  71.696632
2020-01-06  72.267921
2020-01-07  71.928055
2020-01-08  73.085121


In [2]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

def create_dataset(data, time_step=60):
    X, y = [], []
    for i in range(len(data)-time_step):
        X.append(data[i:i+time_step])
        y.append(data[i+time_step])
    return np.array(X), np.array(y)

X, y = create_dataset(scaled_data)

In [3]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(60,1)),
    LSTM(50),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X, y, epochs=5, batch_size=32)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 77ms/step - loss: 0.0462
Epoch 2/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 64ms/step - loss: 0.0038
Epoch 3/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 54ms/step - loss: 0.0019
Epoch 4/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 59ms/step - loss: 0.0016
Epoch 5/5
30/30 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - loss: 0.0015


In [8]:
model.save("stock_model.keras")

In [10]:
import tensorflow as tf

# Load the Keras model
reloaded_model = tf.keras.models.load_model("stock_model.keras")

# Convert the Keras model to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(reloaded_model)

# Apply the suggested fixes for dynamic shapes in LSTM models
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open("stock_model.tflite", "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmpudm54iec'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 60, 1), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  134505003192144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134505003189072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134505003194448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134505003196176: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134505003195792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134505003193488: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134505003194064: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134505003194640: TensorSpec(shape=(), dtype=tf.resource, name=None)
